# Socceraction

## Libraries and Constants

In [7]:
# Import libraries
import pandas as pd
import socceraction.spadl as spadl
import socceraction.vaep.features as fs
import socceraction.vaep.labels as lab

from pathlib import Path
from tqdm import tqdm

In [8]:
# Ignore warnings
import warnings

warnings.filterwarnings(action="ignore", message="Inferred xy_fidelity_version=2.", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [9]:
# Define the data directory
DATA_DIR = Path("../data")
SPADL_DIR = Path("data")

SPADL_H5 = SPADL_DIR / "spadl-statsbomb.h5"
FEATURES_H5 = SPADL_DIR / "features.h5"
LABELS_H5 = SPADL_DIR / "labels.h5"
PREDICTIONS_H5 = SPADL_DIR / "predictions.h5"

## VAEP

In [10]:
# Load the games data from the HDF5 file
games_df = pd.read_hdf(SPADL_H5, "games")
print(f"Number of games: {len(games_df)}")

Number of games: 380


In [11]:
# Create a list of tuples containing game_id and home_team_id
game_home_team_pair = list(games_df[["game_id", "home_team_id"]].itertuples(index=False))
print(*game_home_team_pair[:10], sep="\n")

Pandas(game_id=3754058, home_team_id=22)
Pandas(game_id=3754245, home_team_id=27)
Pandas(game_id=3754136, home_team_id=37)
Pandas(game_id=3754037, home_team_id=29)
Pandas(game_id=3754039, home_team_id=31)
Pandas(game_id=3754041, home_team_id=1)
Pandas(game_id=3754042, home_team_id=27)
Pandas(game_id=3754043, home_team_id=38)
Pandas(game_id=3754045, home_team_id=22)
Pandas(game_id=3754048, home_team_id=31)


In [12]:
# Define the feature functions
xfns = [
    fs.actiontype,
    fs.actiontype_onehot,
    fs.result,
    fs.result_onehot,
    fs.bodypart,
    fs.bodypart_onehot,
    fs.startlocation,
    fs.endlocation,
    fs.startpolar,
    fs.endpolar,
    fs.movement,
    fs.space_delta,
    fs.time,
    fs.time_delta,
    fs.team,
    fs.goalscore,
]

with pd.HDFStore(SPADL_H5) as spadl_store, pd.HDFStore(FEATURES_H5) as feature_store:
    for game_id, home_team_id in tqdm(game_home_team_pair, desc=f"Generating and storing features in {FEATURES_H5}"):
        actions = spadl_store[f"actions/game_{game_id}"]
        actions = spadl.add_names(actions)

        gamestates = fs.gamestates(actions, nb_prev_actions=3)
        gamestates = fs.play_left_to_right(gamestates, home_team_id)

        X = pd.concat([fn(gamestates) for fn in xfns], axis=1)
        feature_store.put(f"game_{game_id}", X, format="table")

Generating and storing features in data\features.h5: 100%|██████████| 380/380 [01:46<00:00,  3.56it/s]


In [13]:
# Define the label functions
yfns = [lab.scores, lab.concedes, lab.goal_from_shot]

with pd.HDFStore(SPADL_H5) as spadl_store, pd.HDFStore(LABELS_H5) as label_store:
    for game_id, _ in tqdm(game_home_team_pair, desc=f"Computing and storing labels in {LABELS_H5}"):
        actions = spadl_store[f"actions/game_{game_id}"]
        actions = spadl.add_names(actions)

        Y = pd.concat([fn(actions) for fn in yfns], axis=1)
        label_store.put(f"game_{game_id}", Y, format="table")

Computing and storing labels in data\labels.h5: 100%|██████████| 380/380 [00:20<00:00, 18.31it/s]
